# Traffic Flow Analyzer — Detection Exploration
Visualize YOLOv8 vehicle detections frame by frame.

In [ ]:
import sys
sys.path.insert(0, '..')  # so 'src' imports work from notebooks/

import cv2
import matplotlib.pyplot as plt
from IPython.display import display
from src.detect import load_model, detect_vehicles, draw_detections

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
VIDEO_PATH = '../data/raw/sample.mp4'   # ← change to your video
CONF       = 0.3
FRAMES_TO_SHOW = 5   # how many sample frames to display

model = load_model('yolov8n.pt')
print('Model loaded ✓')

In [ ]:
# ── Helper: show a BGR frame inline ─────────────────────────────────────────
def show_frame(frame, title=''):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(rgb)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Sample frames from the video ────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Total frames: {total_frames}')

step = max(1, total_frames // FRAMES_TO_SHOW)
shown = 0
frame_idx = 0

while shown < FRAMES_TO_SHOW:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        break

    detections = detect_vehicles(model, frame, conf_threshold=CONF)
    annotated  = draw_detections(frame, detections)

    title = f'Frame {frame_idx} — {len(detections)} vehicle(s) detected'
    show_frame(annotated, title)

    print(f'Frame {frame_idx}: {detections}')

    frame_idx += step
    shown += 1

cap.release()

In [ ]:
# ── Detection count distribution ────────────────────────────────────────────
from collections import Counter

cap = cv2.VideoCapture(VIDEO_PATH)
class_counts = Counter()
frame_counts = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
    dets = detect_vehicles(model, frame, conf_threshold=CONF)
    frame_counts.append(len(dets))
    for d in dets:
        class_counts[d['class_name']] += 1

cap.release()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vehicles per frame over time
axes[0].plot(frame_counts, linewidth=0.8)
axes[0].set_title('Vehicles detected per frame')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Count')

# Total per class
axes[1].bar(class_counts.keys(), class_counts.values(), color=['green','orange','red','magenta'])
axes[1].set_title('Total detections per class')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('Class totals:', dict(class_counts))